<a href="https://colab.research.google.com/github/DrAbdulmalek/DrAbdulmalek/blob/main/Qwen3_14B_RE_Assistant(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen3-14B Abliterated — مساعد الهندسة العكسية\n\n1. فعّل GPU: Runtime → Change runtime type → T4\n2. شغّل الخلايا بالترتيب

In [2]:
# ========== الخلية 1: الإعداد + Google Drive + تحميل ذكي ==========
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!pip install -q gradio huggingface_hub pefile pyelftools capstone

from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from huggingface_hub import hf_hub_download

DRIVE_DIR = "/content/drive/MyDrive/LLM_models"
os.makedirs(DRIVE_DIR, exist_ok=True)
FILE = "huihui-ai_Qwen3-14B-abliterated-Q4_K_M.gguf"
local_path = f"/content/{FILE}"
drive_path = f"{DRIVE_DIR}/{FILE}"

if os.path.exists(drive_path):
    print("✅ النموذج موجود في Drive — نسخه للقرص المحلي…")
    shutil.copy(drive_path, local_path)
else:
    print("⬇️ أول مرة: تحميل من HuggingFace (~9GB)…")
    local_path = hf_hub_download(
        repo_id="bartowski/huihui-ai_Qwen3-14B-abliterated-GGUF",
        filename=FILE
    )
    shutil.copy(local_path, drive_path)
    print("💾 تم الحفظ في Google Drive")

print(f"📦 جاهز: {local_path}")

Mounted at /content/drive
⬇️ أول مرة: تحميل من HuggingFace (~9GB)…


huihui-ai_Qwen3-14B-abliterated-Q4_K_M.g(…): reconstructing file:   0%|          |  0.00B / 9.00GB            

huihui-ai_Qwen3-14B-abliterated-Q4_K_M.g(…): downloading bytes:           |  0.00B            

💾 تم الحفظ في Google Drive
📦 جاهز: /root/.cache/huggingface/hub/models--bartowski--huihui-ai_Qwen3-14B-abliterated-GGUF/snapshots/623c0f3fc42a4699d4583fb16e022942c003d1b7/huihui-ai_Qwen3-14B-abliterated-Q4_K_M.gguf


In [3]:
# ========== الخلية 2: تحميل النموذج في الذاكرة ==========
from llama_cpp import Llama

llm = Llama(
    model_path=local_path,
    n_ctx=8192,
    n_gpu_layers=-1,   # إذا امتلأت VRAM: غيّر إلى 25
    chat_format="chatml"
)
print("✅ النموذج محمّل")

RuntimeError: Failed to load shared library '/usr/local/lib/python3.13/dist-packages/llama_cpp/lib/libllama.so': libcudart.so.12: cannot open shared object file: No such file or directory

In [ ]:
# ========== الخلية 3: واجهة الدردشة + رفع ملفات + تحليل PE/ELF كامل ==========
import gradio as gr
import os, re

SYSTEM = """You are an expert reverse engineering assistant.
Analyze binary files, hexdumps, strings, imports/exports and disassembly.
Identify compilers/packers and suggest Ghidra/IDA next steps."""

_state = {"last_file": None, "analysis": None}

def detect_type(data):
    if data[:2] == b'MZ': return "PE (Windows)"
    if data[:4] == b'\x7fELF': return "ELF (Linux)"
    if data[:2] == b'PK': return "ZIP/Archive"
    if data[:2] == b'\x1f\x8b': return "Gzip"
    return "Unknown"

# ---------- PE: معلومات عامة ----------
def pe_info(path):
    out = []
    try:
        import pefile
        pe = pefile.PE(path, fast_load=True)
        pe.parse_data_directories()
        out.append(f"Machine: {'x64' if pe.FILE_HEADER.Machine == 0x8664 else 'x86'}")
        out.append(f"EntryPoint: 0x{pe.OPTIONAL_HEADER.AddressOfEntryPoint:x}")
        out.append(f"ImageBase: 0x{pe.OPTIONAL_HEADER.ImageBase:x}")
        out.append(f"Subsystem: {'GUI' if pe.OPTIONAL_HEADER.Subsystem == 2 else 'Console'}")
        out.append(f"Sections: {', '.join(s.Name.decode(errors='replace').strip() for s in pe.sections)}")
    except Exception as e:
        out.append(f"PE parse error: {e}")
    return "\n".join(out)

# ---------- PE: استخراج الـ Imports/Exports الكامل ----------
def pe_imports_exports(path, max_funcs_per_dll=15, max_dlls=30):
    out = []
    try:
        import pefile
        pe = pefile.PE(path, fast_load=True)
        pe.parse_data_directories()
        # IMPORTS
        imports = getattr(pe, 'DIRECTORY_ENTRY_IMPORT', [])
        if imports:
            out.append(f"### IMPORTS ({len(imports)} DLLs)")
            for entry in imports[:max_dlls]:
                dll = entry.dll.decode(errors='replace')
                funcs = []
                for imp in entry.imports[:max_funcs_per_dll]:
                    funcs.append(imp.name.decode(errors='replace') if imp.name else f"ord:{imp.ordinal}")
                more = f" (+{len(entry.imports)-max_funcs_per_dll} more)" if len(entry.imports) > max_funcs_per_dll else ""
                out.append(f"  {dll}: {', '.join(funcs)}{more}")
        else:
            out.append("### IMPORTS: (none)")
        # EXPORTS
        if hasattr(pe, 'DIRECTORY_ENTRY_EXPORT'):
            exps = pe.DIRECTORY_ENTRY_EXPORT.symbols
            out.append(f"### EXPORTS ({len(exps)} functions)")
            for e in exps[:40]:
                name = e.name.decode(errors='replace') if e.name else f"ord:{e.ordinal}"
                out.append(f"  {name} @ RVA 0x{e.address:x} ord:{e.ordinal}")
        else:
            out.append("### EXPORTS: (none — not a DLL or no export table)")
    except Exception as e:
        out.append(f"imports/exports error: {e}")
    return "\n".join(out)

# ---------- ELF: معلومات عامة ----------
def elf_info(path):
    out = []
    try:
        from elftools.elf.elffile import ELFFile
        with open(path, 'rb') as f:
            elf = ELFFile(f)
            out.append(f"Arch: {elf.get_machine_arch()}, Entry: 0x{elf.header.e_entry:x}")
            out.append(f"Sections: {', '.join(s.name for s in elf.iter_sections())}")
    except Exception as e:
        out.append(f"ELF parse error: {e}")
    return "\n".join(out)

# ---------- ELF: استخراج الـ Imports/Exports الكامل ----------
def elf_imports_exports(path, max_syms=60):
    out = []
    try:
        from elftools.elf.elffile import ELFFile
        with open(path, 'rb') as f:
            elf = ELFFile(f)
            # IMPORTS: undefined symbols in .dynsym
            dynsym = elf.get_section_by_name('.dynsym')
            if dynsym:
                undefined, defined = [], []
                for sym in dynsym.iter_symbols():
                    if sym['st_shndx'] == 'SHN_UNDEF' and sym.name:
                        undefined.append(sym.name)
                    elif sym.name:
                        defined.append((sym.name, hex(sym['st_value'])))
                out.append(f"### IMPORTS (undefined dynamic symbols, {len(undefined)} total)")
                for s in undefined[:max_syms]:
                    out.append(f"  U {s}")
                if len(undefined) > max_syms:
                    out.append(f"  ... +{len(undefined)-max_syms} more")
                # EXPORTS: defined symbols exported (from .symtab or .dynsym)
                out.append(f"### EXPORTS ({len(defined)} defined)")
                for name, addr in defined[:40]:
                    out.append(f"  {addr} {name}")
            else:
                out.append("### no .dynsym section")
            # Libraries needed
            dyn = elf.get_section_by_name('.dynamic')
            if dyn:
                libs = [t.needed for t in dyn.iter_tags() if t.entry.d_tag == 'DT_NEEDED']
                out.append(f"### NEEDED LIBRARIES: {', '.join(libs) if libs else '(none)'}")
    except Exception as e:
        out.append(f"imports/exports error: {e}")
    return "\n".join(out)

# ---------- Disassembly لنقطة الدخول (PE) ----------
def disasm_pe(path):
    out = []
    try:
        import pefile, capstone
        pe = pefile.PE(path)
        ep = pe.OPTIONAL_HEADER.AddressOfEntryPoint
        data = pe.get_memory_mapped_image()[ep:ep+128]
        mode = capstone.CS_MODE_64 if pe.FILE_HEADER.Machine == 0x8664 else capstone.CS_MODE_32
        for i in capstone.Cs(capstone.CS_ARCH_X86, mode).disasm(data, ep + pe.OPTIONAL_HEADER.ImageBase):
            out.append(f"0x{i.address:x}: {i.mnemonic} {i.op_str}")
            if len(out) >= 25: break
    except Exception as e:
        out.append(f"disasm error: {e}")
    return "\n".join(out) if out else "(none)"

def hexdump(data, length=512):
    lines = []
    for i in range(0, min(length, len(data)), 16):
        c = data[i:i+16]
        lines.append(f"{i:08x}  {' '.join(f'{b:02x}' for b in c):<48}  {''.join(chr(b) if 32<=b<127 else '.' for b in c)}")
    return "\n".join(lines)

def analyze_file(file):
    if file is None: return None
    path = file.name if hasattr(file, 'name') else str(file)
    if _state["last_file"] == path: return _state["analysis"]
    with open(path, 'rb') as f: data = f.read()
    ftype = detect_type(data)
    strings = re.findall(rb'[\x20-\x7e]{6,}', data[:200_000])
    strings_out = "\n".join(s.decode() for s in strings[:80]) or "(none)"
    parts = [f"=== FILE ANALYSIS ===\nType: {ftype}\nSize: {len(data):,} bytes\nMagic: {data[:16].hex()}\n",
             "--- HEADER INFO ---"]
    if "PE" in ftype:
        parts += [pe_info(path), "--- IMPORTS/EXPORTS ---", pe_imports_exports(path),
                  "--- ENTRY POINT DISASM ---", disasm_pe(path)]
    elif "ELF" in ftype:
        parts += [elf_info(path), "--- IMPORTS/EXPORTS ---", elf_imports_exports(path)]
    parts += ["--- HEXDUMP (first 512 bytes) ---", hexdump(data),
              "--- STRINGS (top 80) ---", strings_out,
              "=====================\nProvide deep RE analysis: compiler/packer, suspicious imports (e.g. WinAPI usage patterns), interesting exports, next steps in Ghidra/IDA."]
    _state.update(last_file=path, analysis="\n".join(parts))
    return _state["analysis"]

def chat(message, history, file):
    messages = [{"role": "system", "content": SYSTEM}]
    analysis = analyze_file(file)
    if analysis:
        messages += [{"role": "user", "content": analysis},
                     {"role": "assistant", "content": "File analyzed with full imports/exports. Ask me anything about it."}]
    for h in history:
        messages += [{"role": "user", "content": h[0]}, {"role": "assistant", "content": h[1]}]
    if message:
        messages.append({"role": "user", "content": message})
    return llm.create_chat_completion(messages=messages, max_tokens=2048, temperature=0.7)["choices"][0]["message"]["content"]

with gr.Blocks(title="RE Assistant — Qwen3-14B Abliterated") as demo:
    gr.Markdown("# 🔍 مساعد الهندسة العكسية\nارفع ملفاً ثم اسأل عنه، أو دردش مباشرة")
    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(label="ملف ثنائي", file_types=[".exe", ".dll", ".elf", ".bin", ".so", ".sys"])
        with gr.Column(scale=3):
            gr.ChatInterface(chat, additional_inputs=[file_input], title="")
demo.launch(share=True)